In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# 1. Load the Dataset
file_path = 'bbc-news-data.csv'

if os.path.exists(file_path):
    # Load the CSV (using sep='\t' because some BBC Kaggle files use tabs)
    # If this gives an error, change it to pd.read_csv(file_path)
    try:
        df = pd.read_csv(file_path, sep='\t')
    except:
        df = pd.read_csv(file_path)

    # Check column names: Kaggle BBC files usually use 'content' and 'category'
    # We rename them to 'text' and 'label' for simplicity
    if 'content' in df.columns:
        df = df.rename(columns={'content': 'text', 'category': 'label'})
    elif 'text' in df.columns:
         df = df.rename(columns={'text': 'text', 'category': 'label'})

    # --- THE SAFETY SHIELD (Fixes NaN Errors) ---
    df['text'] = df['text'].astype(str).fillna('')
    df = df[df['text'].str.strip() != ""]
    df = df.dropna(subset=['text', 'label'])

    # 2. Split Data (80% Train, 20% Test)
    X_train, X_test, y_train, y_test = train_test_split(
        df['text'], df['label'], test_size=0.2, random_state=42
    )

    # 3. Build the Pipeline
    # CountVectorizer turns text to numbers; MultinomialNB is the classifier
    model = make_pipeline(CountVectorizer(stop_words='english'), MultinomialNB())

    # 4. Train the Model
    print(f"Training on {len(X_train)} news articles...")
    model.fit(X_train, y_train)

    # 5. Accuracy Check
    accuracy = model.score(X_test, y_test)
    print(f"✅ Success! Model Accuracy: {accuracy * 100:.2f}%")

    # 6. Test with a custom headline
    print("-" * 30)
    sample_news = ["The new smartphone features a powerful processor and 5G connectivity."]
    prediction = model.predict(sample_news)

    print(f"Input: {sample_news[0]}")
    print(f"Predicted Category: {prediction[0].upper()}")

else:
    print(f"Error: '{file_path}' not found in the compiler sidebar.")
    print("Please ensure you have uploaded 'bbc-news-data.csv'.")